# Tulika's GTEx isoforms that Tim did not rescore: missing variants only

Builds the list of (variant, isoform) pairs to send back to Tim for rescoring. Only the **missing** variants
are used; the 1,000 controls are excluded. (The earlier version that also includes the controls is
`tulika_gtex_isoforms_not_rescored_by_tim.ipynb`.)

**Steps**
1. Start from the request sent to Tim (`mp2_request_for_tim.tsv`) and keep only the 21,737 missing variants.
2. Keep the rows of Tulika's file whose variant is on that list and that have a score, one row per
   (variant, isoform).
3. Drop Tulika's rows whose isoform is **not** in the GTEx expression file.
4. Of what remains, keep the (variant, isoform) pairs Tim did **not** return a score for.
5. Write the result to `data/AI_generated/tulika_gtex_isoforms_not_rescored_by_tim_missing_only.tsv`.

**Isoform matching.** Everywhere in this notebook, isoforms are compared on the Ensembl ID **without** the
version suffix (`ENST00000369519.4` → `ENST00000369519`), because Tulika's version numbers often differ from
the GTEx / Tim versions for the same isoform.

**Tulika's scores** are carried along only as a reference, to show what she had. They are not used as scores.

## 1. Setup

In [1]:
from pathlib import Path

import pandas as pd

from ExTSP.config import DATA_DIR, GTeX_file

DATA        = Path(str(DATA_DIR))
REQUEST     = DATA / "AI_generated" / "mp2_request_for_tim.tsv"
TIM_RETURN  = DATA / "MutPred2_scores" / "MutPred2_scores_missing_and_control.tsv"
TULIKA_FILE = DATA / "MutPred2_scores" / "mutpred_score_variant_transcript_tulika.tsv"
OUT         = DATA / "AI_generated" / "tulika_gtex_isoforms_not_rescored_by_tim_missing_only.tsv"

for f in (REQUEST, TIM_RETURN, TULIKA_FILE, GTeX_file):
    print(f"{'OK ' if Path(f).exists() else 'MISSING'}  {f}")


def strip_version(ids):
    """ENST00000369519.4 -> ENST00000369519"""
    return ids.str.split(".").str[0]

OK   /workspaces/ExTSP-default-claude/src/ExTSP/data/AI_generated/mp2_request_for_tim.tsv
OK   /workspaces/ExTSP-default-claude/src/ExTSP/data/MutPred2_scores/MutPred2_scores_missing_and_control.tsv
OK   /workspaces/ExTSP-default-claude/src/ExTSP/data/MutPred2_scores/mutpred_score_variant_transcript_tulika.tsv
OK   /workspaces/ExTSP-default-claude/src/ExTSP/data/raw/GTeX_V8_short_read_transcripts_meanTPM_expression_across_tissue_samples_Gencodev26_without_tpm_filter.tsv


## 2. Step 1: the missing variants

`missing = True` marks the 21,737 variants SJ had no score for; `False` marks the 1,000 controls, which are
dropped here.

In [2]:
request = pd.read_csv(REQUEST, sep="\t")
print(f"request variants : {len(request)}  (duplicated variants: {request['Variant'].duplicated().sum()})")
print(request["missing"].map({True: "missing", False: "control"}).value_counts().to_string())

missing = request[request["missing"]]
print(f"\nkept (missing only): {len(missing)}")
missing.head()

request variants : 22737  (duplicated variants: 0)
missing
missing    21737
control     1000

kept (missing only): 21737


,Variant,sheets,Tulika,SJ,missing,consequence
0,10_100806598_C/A,nonTarget,False,False,True,nonsense
1,10_100988226_C/T,nonTarget,False,False,True,nonsense
2,10_100989063_C/T,nonTarget,False,False,True,nonsense
3,10_101016407_G/A,nonTarget,False,False,True,nonsense
4,10_101021908_T/A,nonTarget,False,False,True,nonsense


## 3. Step 2: map Tulika's file onto the missing variants

Tulika's file is one row per variant x isoform: `mutation` is the variant (same `chr_pos_REF/ALT` format as
the request), `Feature` is her versioned isoform, `mutpred` is her score.

Filters, in order:
- keep rows whose variant is on the missing list
- drop rows with no score
- keep **one row per (variant, isoform ID)**. Her file repeats many rows word for word, and a few
  (variant, isoform) pairs appear twice with scores that differ only in rounding (e.g. 0.228 vs 0.22845).
  The first row in file order is kept. The isoform ID is taken without the version, so the same isoform
  listed under two versions also counts as a duplicate.

In [3]:
tulika_raw = pd.read_csv(TULIKA_FILE, sep="\t")
print(f"Tulika file              : {len(tulika_raw)} rows, {tulika_raw['mutation'].nunique()} variants")

tu = tulika_raw[tulika_raw["mutation"].isin(missing["Variant"])]
print(f"  on the missing list    : {len(tu)} rows, {tu['mutation'].nunique()} variants")

tu = tu[tu["mutpred"].notna()]
print(f"  with a score           : {len(tu)} rows, {tu['mutation'].nunique()} variants")

tu = tu.rename(columns={"mutation": "Variant", "external_gene_name": "gene",
                        "Feature": "transcript_tulika", "mutpred": "MutPred2_tulika_reference"})
tu["isoform_id"] = strip_version(tu["transcript_tulika"])

Tulika file              : 632653 rows, 96253 variants
  on the missing list    : 98620 rows, 12219 variants
  with a score           : 98620 rows, 12219 variants


How the duplicates break down, before removing them:

In [4]:
KEY = ["Variant", "isoform_id"]

per_pair = tu.groupby(KEY).agg(rows=("MutPred2_tulika_reference", "size"),
                               scores=("MutPred2_tulika_reference", "nunique"),
                               versions=("transcript_tulika", "nunique"))
print(f"exact repeated rows                      : {tu.duplicated().sum()}")
print(f"(variant, isoform) pairs                 : {len(per_pair)}")
print(f"  appearing more than once               : {(per_pair['rows'] > 1).sum()}")
print(f"  with more than one score               : {(per_pair['scores'] > 1).sum()}")
print(f"  with more than one isoform version     : {(per_pair['versions'] > 1).sum()}")

conflicts = per_pair[per_pair["scores"] > 1].index
tu.set_index(KEY).loc[conflicts].sort_index()[["transcript_tulika", "Substitution", "MutPred2_tulika_reference"]]

exact repeated rows                      : 49310
(variant, isoform) pairs                 : 49303
  appearing more than once               : 49303
  with more than one score               : 7
  with more than one isoform version     : 0


transcript_tulika Substitution  \
Variant         isoform_id                                        
20_31821684_T/C ENST00000375994  ENST00000375994.6        V240A   
                ENST00000375994  ENST00000375994.6        V240A   
                ENST00000375994  ENST00000375994.6        V240A   
                ENST00000375994  ENST00000375994.6        V240A   
3_38606773_C/T  ENST00000414099  ENST00000414099.6        E346K   
                ENST00000414099  ENST00000414099.6        E346K   
                ENST00000414099  ENST00000414099.6        E346K   
                ENST00000414099  ENST00000414099.6        E346K   
                ENST00000449557  ENST00000449557.6        E346K   
                ENST00000449557  ENST00000449557.6        E346K   
                ENST00000449557  ENST00000449557.6        E346K   
                ENST00000449557  ENST00000449557.6        E346K   
                ENST00000450102  ENST00000450102.6        E346K   
                ENST00000450102  ENST00000450102.6        E346K   
                ENST00000450102  ENST00000450102.6        E346K   
                ENST00000450102  ENST00000450102.6        E346K   
                ENST00000455624  ENST00000455624.6        E346K   
                ENST00000455624  ENST00000455624.6        E346K   
                ENST00000455624  ENST00000455624.6        E346K   
                ENST00000455624  ENST00000455624.6        E346K   
5_225938_G/A    ENST00000504309  ENST00000504309.5        R171H   
                ENST00000504309  ENST00000504309.5        R171H   
                ENST00000504309  ENST00000504309.5        R171H   
                ENST00000504309  ENST00000504309.5        R171H   
                ENST00000510361  ENST00000510361.5        R123H   
                ENST00000510361  ENST00000510361.5        R123H   
                ENST00000510361  ENST00000510361.5        R123H   
                ENST00000510361  ENST00000510361.5        R123H   

                                 MutPred2_tulika_reference  
Variant         isoform_id                                  
20_31821684_T/C ENST00000375994                   0.090469  
                ENST00000375994                   0.090000  
                ENST00000375994                   0.090469  
                ENST00000375994                   0.090000  
3_38606773_C/T  ENST00000414099                   0.595550  
                ENST00000414099                   0.596000  
                ENST00000414099                   0.595550  
                ENST00000414099                   0.596000  
                ENST00000449557                   0.630900  
                ENST00000449557                   0.631000  
                ENST00000449557                   0.630900  
                ENST00000449557                   0.631000  
                ENST00000450102                   0.591080  
                ENST00000450102                   0.591000  
                ENST00000450102                   0.591080  
                ENST00000450102                   0.591000  
                ENST00000455624                   0.588050  
                ENST00000455624                   0.588000  
                ENST00000455624                   0.588050  
                ENST00000455624                   0.588000  
5_225938_G/A    ENST00000504309                   0.751570  
                ENST00000504309                   0.752000  
                ENST00000504309                   0.751570  
                ENST00000504309                   0.752000  
                ENST00000510361                   0.811160  
                ENST00000510361                   0.811000  
                ENST00000510361                   0.811160  
                ENST00000510361                   0.811000

In [5]:
n = len(tu)
tu = tu.drop_duplicates(KEY, keep="first")
assert not tu.duplicated(KEY).any()
print(f"rows removed, keeping one per (variant, isoform): {n - len(tu)}  -> {len(tu)} rows")

tu = tu.merge(missing[["Variant", "sheets", "consequence"]], on="Variant", how="left")
print(f"missing variants Tulika has a score for: {tu['Variant'].nunique()}")

rows removed, keeping one per (variant, isoform): 49317  -> 49303 rows
missing variants Tulika has a score for: 12219


## 4. Step 3: drop Tulika's isoforms that are not in GTEx

The GTEx expression file lists 158 isoforms twice: once normally and once with a `_PAR_Y` suffix (their copy
in the region shared by the X and Y chromosomes). The `_PAR_Y` copies are ignored, so each isoform ID maps to
exactly one GTEx version. That version (`transcript_gtex`) is kept in the output, because it's the version Tim's
run uses.

In [6]:
gtex_ids = pd.read_csv(GTeX_file, sep="\t", usecols=["transcript_id"])["transcript_id"]
gtex_ids = gtex_ids[~gtex_ids.str.endswith("_PAR_Y")]
gtex = pd.DataFrame({"isoform_id": strip_version(gtex_ids), "transcript_gtex": gtex_ids})
assert not gtex["isoform_id"].duplicated().any()
print(f"GTEx isoforms: {len(gtex)}")

tu = tu.merge(gtex, on="isoform_id", how="left")
in_gtex = tu["transcript_gtex"].notna()

print(f"Tulika rows                  : {len(tu)}")
print(f"  isoform in GTEx  (kept)    : {in_gtex.sum()}")
print(f"  isoform NOT in GTEx (drop) : {(~in_gtex).sum()}  ({tu.loc[~in_gtex, 'isoform_id'].nunique()} distinct isoforms)")

lost = set(tu.loc[~in_gtex, "Variant"]) - set(tu.loc[in_gtex, "Variant"])
print(f"  variants left with no GTEx isoform at all: {len(lost)}")

tu_g = tu[in_gtex].copy()
print(f"\nvariants remaining: {tu_g['Variant'].nunique()}")

GTEx isoforms: 199166
Tulika rows                  : 49303
  isoform in GTEx  (kept)    : 48789
  isoform NOT in GTEx (drop) : 514  (326 distinct isoforms)
  variants left with no GTEx isoform at all: 9

variants remaining: 12210


## 5. Step 4: which of these did Tim not rescore?

From Tim's return, only rows with an actual score count; a row with a blank score counts as not returned.
Tim's file uses `ID` for the variant.

For each remaining Tulika row:
- `tim_scored_no_isoform = True`: Tim returned no scored isoform at all for this variant
- `tim_scored_no_isoform = False`: Tim scored other isoforms of this variant, but not this one

In [7]:
tim = pd.read_csv(TIM_RETURN, sep="\t")
print(f"Tim rows: {len(tim)}, blank score: {tim['MutPred2 score'].isna().sum()} (not counted)")
tim = tim[tim["MutPred2 score"].notna()]

tim_pairs    = set(zip(tim["ID"], strip_version(tim["transcript"])))
tim_variants = set(tim["ID"])

tu_g["tim_returned_isoform"] = [p in tim_pairs for p in zip(tu_g["Variant"], tu_g["isoform_id"])]
tu_g["tim_scored_no_isoform"] = ~tu_g["Variant"].isin(tim_variants)

print(f"Tulika GTEx rows                 : {len(tu_g)}")
print(f"  Tim returned the same isoform  : {tu_g['tim_returned_isoform'].sum()}")
print(f"  Tim did NOT return it          : {(~tu_g['tim_returned_isoform']).sum()}")

Tim rows: 51101, blank score: 2 (not counted)


Tulika GTEx rows                 : 48789
  Tim returned the same isoform  : 28335
  Tim did NOT return it          : 20454


In [8]:
gap = tu_g[~tu_g["tim_returned_isoform"]]

(gap.groupby("tim_scored_no_isoform")
    .agg(variants=("Variant", "nunique"), rows=("Variant", "size")))

,variants,rows
tim_scored_no_isoform,,
False,4697,18985
True,378,1469


## 6. Step 5: write the file

One row per (variant, isoform) that Tim did not rescore. `transcript_gtex` is the GTEx-versioned ID to send to
Tim; `transcript_tulika` is the version as it appears in her file.

In [9]:
out = (gap[["Variant", "gene", "transcript_gtex", "transcript_tulika", "Substitution",
            "MutPred2_tulika_reference", "tim_scored_no_isoform", "sheets", "consequence"]]
         .sort_values(["Variant", "transcript_gtex"])
         .reset_index(drop=True))

assert not out.duplicated(["Variant", "transcript_gtex"]).any()
out.to_csv(OUT, sep="\t", index=False)
print(f"wrote {len(out)} rows -> {OUT}")
out.head(10)

wrote 20454 rows -> /workspaces/ExTSP-default-claude/src/ExTSP/data/AI_generated/tulika_gtex_isoforms_not_rescored_by_tim_missing_only.tsv


,Variant,gene,transcript_gtex,transcript_tulika,Substitution,MutPred2_tulika_reference,tim_scored_no_isoform,sheets,consequence
0,10_110820072_G/A,RBM20,ENST00000369519.3,ENST00000369519.4,A851T,0.034,True,CM,missense
1,10_119651854_A/G,BAG3,ENST00000369085.7,ENST00000369085.8,K60R,0.142,True,CM,missense
2,10_119669851_G/A,BAG3,ENST00000369085.7,ENST00000369085.8,E61K,0.149,True,CM,missense
3,10_119669851_G/A,BAG3,ENST00000450186.1,ENST00000450186.1,E3K,0.099,True,CM,missense
4,10_119670177_G/C,BAG3,ENST00000369085.7,ENST00000369085.8,E169D,0.072,True,CM,missense
5,10_119670177_G/C,BAG3,ENST00000450186.1,ENST00000450186.1,E111D,0.029,True,CM,missense
6,10_119672255_C/T,BAG3,ENST00000369085.7,ENST00000369085.8,R170W,0.083,True,CM,missense
7,10_119672255_C/T,BAG3,ENST00000450186.1,ENST00000450186.1,R112W,0.042,True,CM,missense
8,10_119672654_C/G,BAG3,ENST00000369085.7,ENST00000369085.8,Q303E,0.238,True,CM,missense
9,10_119672654_C/G,BAG3,ENST00000450186.1,ENST00000450186.1,Q245E,0.150,True,CM,missense


## 7. Summary

In [10]:
v = out.drop_duplicates("Variant")
print(f"{len(out)} rows, {len(v)} variants, {out['transcript_gtex'].nunique()} isoforms")
print(f"  Tim has no isoform for the variant : {v['tim_scored_no_isoform'].sum():>5} variants, "
      f"{out['tim_scored_no_isoform'].sum():>6} rows")
print(f"  Tim scored other isoforms          : {(~v['tim_scored_no_isoform']).sum():>5} variants, "
      f"{(~out['tim_scored_no_isoform']).sum():>6} rows")

20454 rows, 5075 variants, 280 isoforms
  Tim has no isoform for the variant :   378 variants,   1469 rows
  Tim scored other isoforms          :  4697 variants,  18985 rows
